<a href="https://colab.research.google.com/github/law-ucsdgit/greenshirt-redshirt/blob/main/Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import nested_scopes
import IPython
import numpy as np
import random

player1_mask = 0b0
player2_mask = 0b0
steps_done = 0
currentPlayer = 0
state = np.zeros((7,7))
def updateState(action):
  global state, player1_mask, player2_mask, currentPlayerMask, currentPlayer
  blocked = False
  ns = state.clone().detach()
  for row_idx in range(state.shape[1] - 1, 0, -1):
    if state[0, row_idx, action] == 0:
      ns[0, row_idx, action] = currentPlayer * 2 - 1

      #pretends like the enemy played the move
      changeMask = player2_mask if currentPlayer == 0 else player1_mask
      if currentPlayer == 1:
        player1_mask |= (1 << (action * 7 + row_idx))
      else:
        player2_mask |= (1 << (action * 7 + row_idx))

      #checks if the enemy played this move, they would win, which means the bot has blocked a win
      if check_win():
        # print("blocked", ns, currentPlayer, action)
        blocked = True
      if currentPlayer == 1:
        player1_mask = changeMask
      else:
        player2_mask = changeMask


      if currentPlayer == 0:
        player1_mask |= (1 << (action * 7 + row_idx))
      else:
        player2_mask |= (1 << (action * 7 + row_idx))
      currentPlayer = 1 - currentPlayer
      return (True, blocked, ns)
  ns[0, 0, action] = currentPlayer * 2 - 1
  # print("broke/cant move")
  return (False, blocked, ns)
def check_win():
  """Checks if the player who just moved won the game using bitwise shifts."""
  # Check directions: Horizontal (7), Vertical (1), Diagonal \ (6), Diagonal / (8)
  directions = [7, 1, 6, 8]
  if currentPlayer == 1:
    mask = player1_mask
  else:
    mask = player2_mask

  for d in directions:
      # Shift pieces over and check where two adjacent tokens exist
      step1 = mask & (mask >> d)
      # Shift over again to check where four adjacent tokens connect
      if (step1 & (step1 >> (2 * d))) != 0:
          display(IPython.display.Javascript(f'''
        document.querySelector("#debug-text-area").textContent = "win";
    '''))
          return True
  display(IPython.display.Javascript(f'''
  document.querySelector("#debug-text-area").textContent = "{mask}";
    '''))
  return False



class rlEnvironment():
  steps = 0

  def __init__(self):
    global state, currentPlayer, player1_mask, player2_mask
    currentPlayer = 0
    player1_mask = 0b0
    player2_mask = 0b0
    self.action_space = [0, 1, 2, 3, 4, 5, 6]
    state = np.zeros((7,7))
  def reset(self):
    global state, currentPlayer, player1_mask, player2_mask
    currentPlayer = 0
    player1_mask = 0b0
    player2_mask = 0b0
    state = np.zeros((7,7))
    return state
  def step(self, action):
    couldUpdate, self.blocked, ns = updateState(action)
    reward = 0
    if couldUpdate == False:
      reward = -100
      done = True
      return ns, reward, done, False, {}
    done = check_win()
    if self.blocked:
      reward += 3
    if done:
      reward += 100
    if action == 3:
      reward += 0.1
    self.steps += 1
    return ns, reward, done, False, {}

In [ ]:
BATCH_SIZE = 10
EPS_START = 0.9
EPS_END = 0.1
GAMMA = 0.99
EPS_DECAY = 10000
TAU = 0.002
LR = 0.001

In [ ]:


from collections import namedtuple, deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math

Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))
env = rlEnvironment()

def select_action(state):
    global steps_done
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        with torch.no_grad():
            # t.max(1) will return the largest column value of each row.
            # second column on max result is index of where max element was
            # found, so we pick action with the larger expected reward.
            if currentPlayer == 0:
              return policy_net1(state.view(1, 49)).max(1).indices.view(1, 1)
            else:
              tempstate = state.clone().detach()
              tempstate *= -1
              return policy_net2(tempstate.view(1, 49)).max(1).indices.view(1, 1)
    else:
        return torch.tensor([[random.choice(env.action_space)]], dtype=torch.long)

class ReplayMemory(object):

    def __init__(self, capacity):
        self.memory = deque([], maxlen=capacity)
    def reset(self, capacity):
        self.memory = deque([], maxlen=capacity)
    def push(self, *args):
        """Save a transition"""
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)

class DQN(nn.Module):

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    # Called with either one element to determine next action, or a batch
    # during optimization. Returns tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)
#every policy_net should read -1 as their move and 1 as enemy move
policy_net1 = DQN(49, 7)
target_net1 = DQN(49, 7)
policy_net2 = DQN(49, 7)
target_net2 = DQN(49, 7)
target_net1.load_state_dict(policy_net1.state_dict())
target_net2.load_state_dict(policy_net2.state_dict())

optimizer1 = optim.AdamW(policy_net1.parameters(), lr=LR, amsgrad=True)
optimizer2 = optim.AdamW(policy_net2.parameters(), lr=LR, amsgrad=True)

memory1 = ReplayMemory(10000)
memory2 = ReplayMemory(10000)

In [ ]:
def optimize_model():
    if len(memory) < BATCH_SIZE:
        return
    transitions = memory.sample(BATCH_SIZE)
    # Transpose the batch (see https://stackoverflow.com/a/19343/3343043 for
    # detailed explanation). This converts batch-array of Transitions
    # to Transition of batch-arrays.
    batch = Transition(*zip(*transitions))
    #print(batch.next_state)
    # Compute a mask of non-final states and concatenate the batch elements
    # (a final state would've been the one after which simulation ended)
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                          batch.next_state)), dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state
                                                if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Compute Q(s_t, a) - the model computes Q(s_t), then we select the
    # columns of actions taken. These are the actions which would've been taken
    # for each batch state according to policy_net
    state_action_values = policy_net(state_batch.view(len(batch.state),49)).gather(1, action_batch) # 128, 1

    # Compute V(s_{t+1}) for all next states.
    # Expected values of actions for non_final_next_states are computed based
    # on the "older" target_net; selecting their best reward with max(1).values
    # This is merged based on the mask, such that we'll have either the expected
    # state value or 0 in case the state was final.
    next_state_values = torch.zeros(BATCH_SIZE) #128
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states.view(len(non_final_next_states),49)).max(1).values
    # Compute the expected Q values
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Compute Huber loss
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    # Optimize the model
    optimizer.zero_grad()
    loss.backward()
    # In-place gradient clipping
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

In [ ]:
memory1.reset(10000)
memory2.reset(10000)

In [ ]:
import os
import torch

def save_checkpoint(model, optimizer, episode, metric_value, filename):
    checkpoint = {
        'episode': episode,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'metric_value': metric_value
    }
    torch.save(checkpoint, filename)
    print(f"--> Saved checkpoint: {filename} (Metric: {metric_value:.2f})")

In [ ]:
from itertools import count
from collections import deque
import numpy as np

# Trackers
p1_wins = deque(maxlen=100)
p2_wins = deque(maxlen=100)

best_p1_winrate = -1.0
best_p2_winrate = -1.0

SAVE_INTERVAL = 100

for i_episode in range(300):
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
    pstate = None

    # Track who won this episode
    episode_winner = None

    for t in count():
        action = select_action(state)
        observation, reward, terminated, truncated, _ = env.step(action.item())
        reward = torch.tensor([reward])
        done = terminated or truncated
        # ... [Your existing step, replay memory, and optimization code] ...

        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32)
        # Store the transition in memory
        memory = memory1 if currentPlayer == 0 else memory2
        if pstate != None:
          if reward.item() == 100:
            preward = torch.tensor([-100])
          if memory == memory2:
            pstate *= -1
          memory.push(pstate, paction, pnext_state, preward)
        # Move to the next state
        pstate = state
        paction = action
        preward = reward
        pnext_state = next_state
        state = next_state

        # Perform one step of the optimization (on the policy network)
        if currentPlayer == 0:
          policy_net = policy_net1
          target_net = target_net1
          optimizer = optimizer1
        else:
          policy_net = policy_net2
          target_net = target_net2
          optimizer = optimizer2
        optimize_model()

        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
        target_net.load_state_dict(target_net_state_dict)


        if done:
          #push and update the last state which has not been pushed yet
          currentPlayer = 1 - currentPlayer
          if reward.item() >= 100:  # Win condition reached
            episode_winner = currentPlayer
          elif reward.item() == -100: # Illegal move / loss condition
            episode_winner = 1 - currentPlayer
          # Update win history (1 if player won, 0 otherwise)
          p1_wins.append(1 if episode_winner == 0 else 0)
          p2_wins.append(1 if episode_winner == 1 else 0)

          memory = memory1 if currentPlayer == 0 else memory2
          if memory == memory2:
            pstate *= -1
          print(i_episode, pstate, paction, pnext_state, preward, currentPlayer)
          memory.push(pstate, paction, pnext_state, preward)

          #print(i_episode, pstate, paction, pnext_state, preward, currentPlayer)

          # Perform one step of the optimization (on the policy network)
          if currentPlayer == 0:
            policy_net = policy_net1
            target_net = target_net1
            optimizer = optimizer1
          else:
            policy_net = policy_net2
            target_net = target_net2
            optimizer = optimizer2
          optimize_model()

          # Soft update of the target network's weights
          # θ′ ← τ θ + (1 −τ )θ′
          target_net_state_dict = target_net.state_dict()
          policy_net_state_dict = policy_net.state_dict()
          for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key]*TAU + target_net_state_dict[key]*(1-TAU)
          target_net.load_state_dict(target_net_state_dict)
          break

    # --- Checkpoint Evaluation & Saving ---
    if (i_episode + 1) % 50 == 0 and len(p1_wins) >= 50:
        p1_winrate = np.mean(p1_wins)
        p2_winrate = np.mean(p2_wins)

        print(f"\n--- Episode {i_episode + 1} Eval ---")
        print(f"Player 1 Win Rate (Last {len(p1_wins)}): {p1_winrate * 100:.1f}%")
        print(f"Player 2 Win Rate (Last {len(p2_wins)}): {p2_winrate * 100:.1f}%")

        # Save Best Player 1 Model
        if p1_winrate > best_p1_winrate:
            best_p1_winrate = p1_winrate
            save_checkpoint(policy_net1, optimizer1, i_episode, p1_winrate, "best_policy_net1.pt")

        # Save Best Player 2 Model
        if p2_winrate > best_p2_winrate:
            best_p2_winrate = p2_winrate
            save_checkpoint(policy_net2, optimizer2, i_episode, p2_winrate, "best_policy_net2.pt")

    # Periodic checkpoint every N episodes
    if (i_episode + 1) % SAVE_INTERVAL == 0:
        save_checkpoint(policy_net1, optimizer1, i_episode, 0, f"checkpoint_p1_ep{i_episode+1}.pt")
        save_checkpoint(policy_net2, optimizer2, i_episode, 0, f"checkpoint_p2_ep{i_episode+1}.pt")

In [ ]:
torch.save(policy_net1.state_dict(), 'policy_model1_weights.pth')
torch.save(policy_net2.state_dict(), 'policy_model2_weights.pth')
torch.save(target_net1.state_dict(), 'target_model1_weights.pth')
torch.save(target_net2.state_dict(), 'target_model2_weights.pth')
from google.colab import files

# This will prompt your browser to download the file automatically
files.download('policy_model1_weights.pth')
files.download('policy_model2_weights.pth')
files.download('target_model1_weights.pth')
files.download('target_model2_weights.pth')

In [ ]:
"""
Load policy net from checkpoint weight before rendering HTML
"""

from IPython.display import JSON
state = np.zeros((7,7))
state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
realPlayer = 0 #0 or 1 player 0 places down -1 and player 1 places down 1, policynet 1 places down -1 and policynet 2 places down 1
computerPlayer = 1 - realPlayer
policyNet = 2 #1 or 2

loaded_policy_net = DQN(49, 7)
player1_mask = 0b0
player2_mask = 0b0

checkpoint_path = "checkpoint_p1_ep300.pt"  # Or "checkpoint_p2_ep300.pt"

# 3. Load the checkpoint dictionary
# checkpoint = torch.load(checkpoint_path)

# # 4. Load weights into model and set to evaluation mode
# loaded_policy_net.load_state_dict(checkpoint['model_state_dict'])
# loaded_policy_net.eval()

temp_policy_net = loaded_policy_net
temp_policy_net = policy_net1
def select_action_for_play():
  with torch.no_grad():
    # t.max(1) will return the largest column value of each row.
    # second column on max result is index of where max element was
    # found, so we pick action with the larger expected reward.
    tempstate = state.clone().detach()
    if (policyNet == 1 and computerPlayer == 1) or (policyNet == 2 and computerPlayer == 0):
      tempstate = tempstate * -1
    tstring = str(tempstate.tolist()).replace('\n', '')
    display(IPython.display.Javascript(f'''
        document.querySelector("#state-text-area").textContent = "{tstring}";
    '''))
    action = temp_policy_net(tempstate.view(1, 49)).max(1).indices.view(1, 1)
    return action.item()
def updateAction(player, row, action):
  string = str(row) + " " + str(action)
  global currentPlayer, player1_mask, player2_mask

  if player == 0:
    currentPlayer = 1
    player1_mask |= (1 << (action * 7 + row))
  else:
    currentPlayer = 0
    player2_mask |= (1 << (action * 7 + row))
  state[0, row, action] = player * 2 - 1

In [ ]:
from IPython.display import display, HTML
from google.colab import output

output.register_callback('notebook.check_win', check_win)
output.register_callback('notebook.update_action', updateAction)
output.register_callback('notebook.selectAction', select_action_for_play)

# Define the HTML and JavaScript structure
html_code = """
<canvas id="connect4" width="700" height="600" style="border:1px solid #ccc;"></canvas>
<div id="debug-text-area"></div>
<div id="update-text-area"></div>
<div id="state-text-area"></div>
<script>
const canvas = document.getElementById('connect4');
const ctx = canvas.getContext('2d');
const humanPlayer = 0;
const gameOver = false;
player = 0

// Game state tracker for rows filled in each column
const filled = [0, 0, 0, 0, 0, 0, 0];
color = '#ff0000';

function drawBoard() {
    ctx.clearRect(0, 0, 700, 600);
    ctx.fillStyle = '#0055ff'; // Board background
    ctx.fillRect(0, 0, 700, 600);

    // Draw slots
    for (let r = 0; r < 6; r++) {
        for (let c = 0; c < 7; c++) {
            drawCircle(c * 100 + 50, r * 100 + 50, 40, '#ffffff');
        }
    }

}

function drawCircle(x, y, radius, color) {
    ctx.beginPath();
    ctx.arc(x, y, radius, 0, 2 * Math.PI);
    ctx.fillStyle = color;
    ctx.fill();
}

// Click handling logic
canvas.addEventListener('mousedown', async function(event) {
    if (gameOver) return;

    if (player != humanPlayer){
      response = await google.colab.kernel.invokeFunction(
        'notebook.selectAction', // The registered callback name
        [],                          // Positional arguments (Array)
        {}
      );
      const col = parseInt(response.data['text/plain'], 10);
      document.querySelector("#update-text-area").textContent = col;
      // If column is full, do nothing
      if (filled[col] >= 6) {
        alert("invalid move");
        return;
      }
      // Red chip logic (drops from the bottom up: 5 minus current filled count)
      const row = 5 - filled[col];
      google.colab.kernel.invokeFunction(
      'notebook.update_action', // The registered callback name
      [player, row + 1, col],                          // Positional arguments (Array)
      {}
    );
      drawCircle(col * 100 + 50, row * 100 + 50, 40, color);
      color = color === '#ff0000' ? '#ffff00' : '#ff0000';
      // Update filled count
      filled[col]++;
      player = 1 - player
      return;
    }
    const rect = canvas.getBoundingClientRect();
    const x = event.clientX - rect.left;

    // Determine column (0-6)
    const col = Math.floor(x / 100);

    // If column is full, do nothing
    if (filled[col] >= 6){
      alert("invalid move");
      return;

    }


    // Red chip logic (drops from the bottom up: 5 minus current filled count)
    const row = 5 - filled[col];

    google.colab.kernel.invokeFunction(
      'notebook.update_action', // The registered callback name
      [player, row + 1, col],                          // Positional arguments (Array)
      {}
    );
    drawCircle(col * 100 + 50, row * 100 + 50, 40, color);
    response = await google.colab.kernel.invokeFunction(
        'notebook.check_win', // The registered callback name
        [],                          // Positional arguments (Array)
        {}
      );
    const tf = response.data['text/plain'];
    if (tf == "True"){
      alert("Game Over");
      gameOver = true;
    }
    color = color === '#ff0000' ? '#ffff00' : '#ff0000';
    // Update filled count
    player = 1 - player
    filled[col]++;
});

// Initial draw
drawBoard();
</script>
"""

display(HTML(html_code))